# NB-3 — Qwen checkpoint hash pre-check (manifest B.7.1; CPU only)

Computes SHA-256 for the 15 Qwen checkpoints on Google Drive and compares them with the locked inventory
(`FMI_manifests_and_decision_rules_v1_2.md`, B.3). No GPU, no model loading, no data. Reads about 62 GB from Drive,
so it takes a while (typically 15–40 minutes). Writes `MyDrive/FMI/NB3/qwen_hash_precheck.json`; NB-4 refuses to start
unless this report exists and says PASS.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, time
from pathlib import Path
from datetime import datetime, timezone
INVENTORY = [
 {
  "cell": "attention_frozen",
  "seed": 13,
  "best_epoch": 1,
  "wave": "wave4b10",
  "drive_file": "wave4b10_qwen17_corrected_frozen_paths_progress/attention_frozen_seed13_best.safetensors",
  "bytes": 4211252,
  "sha256": "d47383e873f7a830b816fdaba9a5731bb7c4638cbeb843ca635da50413c9558c",
  "regime": "frozen",
  "topology": "attention"
 },
 {
  "cell": "mixer_frozen",
  "seed": 13,
  "best_epoch": 3,
  "wave": "wave4b10",
  "drive_file": "wave4b10_qwen17_corrected_frozen_paths_progress/mixer_frozen_seed13_best.safetensors",
  "bytes": 4211180,
  "sha256": "0b15dba7c9f75f0c200b4b60f8b57012ba0ea1940a1c078d852cc4e49fa434d7",
  "regime": "frozen",
  "topology": "mixer"
 },
 {
  "cell": "attention_frozen",
  "seed": 42,
  "best_epoch": 2,
  "wave": "wave4b13",
  "drive_file": "wave4b13_qwen17_corrected_frozen_seed42_progress/attention_frozen_seed42_best.safetensors",
  "bytes": 4211252,
  "sha256": "3b6516f329d27eadbd993c4f9563be303f5fc853293df87bf5c69fe97c6e27e1",
  "regime": "frozen",
  "topology": "attention"
 },
 {
  "cell": "mixer_frozen",
  "seed": 42,
  "best_epoch": 3,
  "wave": "wave4b13",
  "drive_file": "wave4b13_qwen17_corrected_frozen_seed42_progress/mixer_frozen_seed42_best.safetensors",
  "bytes": 4211180,
  "sha256": "456fc913dfccdb637e00e3f48aa2d78a5e2d10654eb20ac4cb5d2d6a1ce02f11",
  "regime": "frozen",
  "topology": "mixer"
 },
 {
  "cell": "attention_frozen",
  "seed": 71,
  "best_epoch": 2,
  "wave": "wave4b14",
  "drive_file": "wave4b14_qwen17_corrected_frozen_seed71_progress/attention_frozen_seed71_best.safetensors",
  "bytes": 4211252,
  "sha256": "871ce186224563a598645906b08cf3ed9c849cdfb733f4ea353de7abcf676e9b",
  "regime": "frozen",
  "topology": "attention"
 },
 {
  "cell": "mixer_frozen",
  "seed": 71,
  "best_epoch": 3,
  "wave": "wave4b14",
  "drive_file": "wave4b14_qwen17_corrected_frozen_seed71_progress/mixer_frozen_seed71_best.safetensors",
  "bytes": 4211180,
  "sha256": "81d944c261bb5dcffe7cb13dd1f7d5152df87a021a1d483eb63d016f92c8d9f6",
  "regime": "frozen",
  "topology": "mixer"
 },
 {
  "cell": "attention_trainable",
  "seed": 13,
  "best_epoch": 1,
  "wave": "wave4b11",
  "drive_file": "wave4b11_qwen17_corrected_trainable_attention_progress/best_model_path_weights.pt",
  "bytes": 6886644540,
  "sha256": "e1cbb96584eecdbcc4b9efe127bb01a0e1659118158c7ce298aef5d040f42029",
  "regime": "trainable",
  "topology": "attention"
 },
 {
  "cell": "mixer_trainable",
  "seed": 13,
  "best_epoch": 1,
  "wave": "wave4b12",
  "drive_file": "wave4b12_qwen17_corrected_trainable_mixer_progress/best_model_path_weights.pt",
  "bytes": 6886644197,
  "sha256": "ade9bde676492048168522a45dfa5ba1811b5d7485c71f947d404fcac072bee6",
  "regime": "trainable",
  "topology": "mixer"
 },
 {
  "cell": "attention_trainable",
  "seed": 42,
  "best_epoch": 3,
  "wave": "wave4b15",
  "drive_file": "wave4b15_qwen17_corrected_trainable_attention_seed42_progress/best_model_path_weights.pt",
  "bytes": 6886644540,
  "sha256": "462b60feebdea8ec541b4e7a275b1e99a7c108bf90344e1e3cd0cd0819e73ac3",
  "regime": "trainable",
  "topology": "attention"
 },
 {
  "cell": "mixer_trainable",
  "seed": 42,
  "best_epoch": 3,
  "wave": "wave4b16",
  "drive_file": "wave4b16_qwen17_corrected_trainable_mixer_seed42_progress/best_model_path_weights.pt",
  "bytes": 6886644197,
  "sha256": "bcd3792d59db37f126e037353092e3142928d9f7b9f3711663ca1ae084fee613",
  "regime": "trainable",
  "topology": "mixer"
 },
 {
  "cell": "attention_trainable",
  "seed": 71,
  "best_epoch": 3,
  "wave": "wave4b17",
  "drive_file": "wave4b17_qwen17_corrected_trainable_attention_seed71_progress/best_model_path_weights.pt",
  "bytes": 6886644540,
  "sha256": "1b9e597e67dc15f5622c90909944715b338b3b4fcb91c1ddc32c99a6fd7a8c15",
  "regime": "trainable",
  "topology": "attention"
 },
 {
  "cell": "mixer_trainable",
  "seed": 71,
  "best_epoch": 3,
  "wave": "wave4b18",
  "drive_file": "wave4b18_qwen17_corrected_trainable_mixer_seed71_progress/best_model_path_weights.pt",
  "bytes": 6886644197,
  "sha256": "5e8b5317623f510c2dc4c4a80c742a9e12af079a447d76a669264bfc165d1d95",
  "regime": "trainable",
  "topology": "mixer"
 },
 {
  "cell": "no_path",
  "seed": 13,
  "best_epoch": 1,
  "wave": "wave4b8",
  "drive_file": "wave4b8_qwen17_trainable_no_path_progress/best_model_path_weights.pt",
  "bytes": 6882431387,
  "sha256": "dc653687667dffd8be4b3ef6218c51298f2da46b390da22a64bcbb129b1e25c6",
  "regime": "trainable",
  "topology": "none"
 },
 {
  "cell": "no_path",
  "seed": 42,
  "best_epoch": 3,
  "wave": "wave4b19",
  "drive_file": "wave4b19_qwen17_trainable_no_path_seed42_progress/best_model_path_weights.pt",
  "bytes": 6882431387,
  "sha256": "af198e20cd0a2bcb9e99a7d246f84b60b734e360870fa536641e90e9b53ac090",
  "regime": "trainable",
  "topology": "none"
 },
 {
  "cell": "no_path",
  "seed": 71,
  "best_epoch": 3,
  "wave": "wave4b20",
  "drive_file": "wave4b20_qwen17_trainable_no_path_seed71_progress/best_model_path_weights.pt",
  "bytes": 6882431387,
  "sha256": "dfb538ba31aaa3c1e08eee432e95ba0c7f58787ea0ce8424e938fde905dd8b8e",
  "regime": "trainable",
  "topology": "none"
 }
]
DRIVE_ROOT = Path('/content/drive/MyDrive/FPDA'); OUT_DIR = Path('/content/drive/MyDrive/FMI/NB3'); OUT_DIR.mkdir(parents=True, exist_ok=True)
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(64 * 1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()
report, all_pass, t0 = [], True, time.time()
for item in INVENTORY:
    p = DRIVE_ROOT / item['drive_file']; row = {**item, 'exists': p.is_file()}
    if not p.is_file():
        row.update({'bytes_on_drive': None, 'sha256_on_drive': None, 'pass': False}); all_pass = False
    else:
        t = time.time(); row['bytes_on_drive'] = p.stat().st_size; row['sha256_on_drive'] = sha256_file(p); row['seconds'] = round(time.time() - t, 1)
        row['pass'] = (row['bytes_on_drive'] == item['bytes']) and (row['sha256_on_drive'] == item['sha256']); all_pass &= row['pass']
    print(f"{item['cell']:20s} seed {item['seed']:>5} {'PASS' if row['pass'] else 'FAIL'}  {row.get('seconds','')}s")
    report.append(row)
summary = {'gate': 'B.7.1', 'status': 'PASS' if all_pass else 'FAIL', 'checked': len(report), 'utc': datetime.now(timezone.utc).isoformat(), 'total_seconds': round(time.time() - t0, 1), 'rows': report}
(OUT_DIR / 'qwen_hash_precheck.json').write_text(json.dumps(summary, indent=2))
print('\nGATE B.7.1:', summary['status'], '| report:', OUT_DIR / 'qwen_hash_precheck.json')